# KND_04 - Step 5: Prepare the data for machine learning

## What we will do

We will turn our EDA decisions into reusable preprocessing steps:

1. Select information that could be used when an animal arrives.
2. Calculate age and handle unavailable or invalid ages.
3. Standardise category labels and group rare categories.
4. Convert categories to numeric columns and scale age.
5. Save the prepared data and the fitted preprocessing pipeline.

A **pipeline** is a saved sequence of steps. It helps us prepare a new animal's details in the same way as our training records.

**Fit means learn a rule from data. Transform means apply the learned rule.** We fit on training data only. We transform validation and test data without learning from them. We do not train or evaluate a prediction model here.

## Files and libraries

Keep **step3_results** beside this notebook. In Colab, run Notebook 3 first, or upload its three record CSVs and `split_notes.json`. Notebook 4 explains the reasons for our choices but does not modify those CSVs.

Needed libraries: pandas, numpy, scikit-learn, scipy and joblib. If imports fail, run `%pip install pandas numpy scikit-learn scipy joblib` in a separate notebook cell. Restart the kernel if prompted. For exact reproduction later, use the version file this notebook saves.

This notebook creates **step5_results** beside step3_results. It writes its reusable Python helper there automatically. No separate helper download is needed.

Source: [County of Sonoma, Animal Shelter Intake and Outcome](https://data.sonomacounty.ca.gov/Government/Animal-Shelter-Intake-and-Outcome/924a-vesw), California, USA. Counts and input files come from our fixed extract and Steps 2-4.

In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import sys
import numpy as np
import pandas as pd
import sklearn
import scipy
import joblib
from sklearn.base import clone

candidates = [Path("step3_results"), Path("outputs") / "step3_results",
              Path("/content/step3_results"), Path("."), Path("/content")]
needed = ["train_records.csv", "validation_records.csv", "test_records.csv", "split_notes.json"]
STEP3_DIR = next((p.resolve() for p in candidates if all((p / name).exists() for name in needed)), None)
if STEP3_DIR is None:
    raise FileNotFoundError("Run Notebook 3 or upload its three record CSVs AND split_notes.json.")

split_notes = json.loads((STEP3_DIR / "split_notes.json").read_text(encoding="utf-8"))
records, source_hashes = {}, {}
for name in ["train", "validation", "test"]:
    path = STEP3_DIR / f"{name}_records.csv"
    source_hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()
    assert source_hashes[name] == split_notes["output_sha256"][path.name], f"{name}: input changed; review Step 3."
    records[name] = pd.read_csv(path, dtype={"id": "string", "impound_number": "string"})
    assert records[name]["final_split"].eq(name).all()
    assert records[name]["source_row_number"].is_unique

assert [len(records[n]) for n in ["train", "validation", "test"]] == [6096, 1163, 1672]
PROJECT_DIR = STEP3_DIR.parent if STEP3_DIR.name == "step3_results" else STEP3_DIR
RESULTS_DIR = PROJECT_DIR / "step5_results"
RESULTS_DIR.mkdir(exist_ok=True)
print("Loaded training, validation and test inputs. No model scores are calculated here.")
print("Results folder:", RESULTS_DIR)

Loaded training, validation and test inputs. No model scores are calculated here.
Results folder: C:\Users\ishin\Documents\Codex\2026-09-15\we-x20\outputs\step5_results


## 1. Choose and justify the inputs

We use **10 original fields**: type, breed, colour, sex, birth date, intake date, intake type, intake subtype, intake condition and intake jurisdiction.

Two changes from the initial 11-field candidate list:

- **Leave out size for this first pipeline.** The labels mix physical sizes with life stages such as PUPPY and KITTN, and the snapshot does not confirm the size at arrival. This is a cautious choice. Reconsider it only with clear timing evidence and training/validation checks.
- **Keep biological sex, discard neuter/spay status.** MALE and NEUTERED become MALE; FEMALE and SPAYED become FEMALE. Unknown or unrecognised labels become UNKNOWN. We avoid using a procedure status that may have changed during the stay. This does not prove when the original sex entry was recorded.

Why other fields are excluded:

- **Outcome date/type/subtype/condition and final duration:** known after arrival, so they reveal later information.
- **Outcome jurisdiction, ZIP code and location:** source definitions describe the destination, not the intake area.
- **Current kennel:** current location may reflect events after arrival.
- **Animal name:** an identifier, sometimes assigned by staff, with no justified role in this model.
- **Animal ID and admission number:** retain separately for audit and split checks, not prediction.
- **intake_total:** a source field for aggregation, not an arrival characteristic.
- **Target, label dates, split names and generated row numbers:** answers or bookkeeping fields.

The source birth date is approximate. Other animal attributes may also have been updated. The retained feature set is an arrival-time assumption for this prototype, not proof of historical data-entry times. Document this limitation in the report.

## 2. The rules we will apply

**Age:** calculate age at arrival. A missing/unreadable birth date or a birth date after arrival means unavailable age. Use the training median for the animal type; use the overall training median if that type has no known training ages. For example, a dog with unknown age receives the training dogs' median age. Add **age_missing = 1** so the model knows the age was estimated. Valid ages have flag 0. OTHER combines species, so its replacement is a rough estimate.

**Unusual values:** retain valid older ages; we have no evidence that being old is an error. Keep legitimate repeat visits. Earlier notebooks already handled eligibility and ambiguous admissions. We remove no more rows in this step.

**Categories:** trim spaces, standardise letter case and retain UNKNOWN. For breed, colour and intake subtype, labels appearing fewer than 10 times in training are grouped together. Ten is an initial practical setting, not a claim that rarer categories are wrong. New labels also map to the rare/new group. We preserve an explicit unknown slot.

**Month:** calculate two calendar values that place December next to January on a circle. They describe season timing without using the year as a row-like shortcut. This is a proposed useful feature, not proof of a seasonal effect.

**Encoding:** turn categories into separate 0/1 columns. For example, DOG can have its own column instead of being assigned an arbitrary larger number than CAT.

**Scaling:** centre and scale age using training statistics. This helps models such as logistic regression. Trees do not require scaling, but the same numerical representation can be used for our initial comparison. The missing flag stays 0/1 and month values are already bounded.

**Class imbalance:** preserve every label and row. Later we will compare unweighted models with training-only class/sample weighting. We do not duplicate records or manufacture missing values.

## 3. Save the reusable code

The following cell writes **shelter_preprocessing.py** into step5_results. It contains the same rules we just described, so later notebooks and the application can import them.

The code is longer than the earlier cells because it packages the rules for reuse. Read it in three parts: `arrival_features` calculates values; `ShelterFeatures.fit` learns medians and category groups; `make_preprocessor` connects these to encoding and scaling.

The encoder reserves unknown/new-category columns without adding any artificial training rows. A new category has no directly learned model evidence, even though the pipeline can represent it.

In [2]:
MODULE_CODE = r'''
"""KND_04 reusable preprocessing. Fit on training admissions only."""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

RAW_INPUT_FIELDS = [
    "type", "breed", "color", "sex", "date_of_birth", "intake_date",
    "intake_type", "intake_subtype", "intake_condition", "intake_jurisdiction",
]
CATEGORICAL_FIELDS = [
    "type", "breed", "color", "biological_sex", "intake_type",
    "intake_subtype", "intake_condition", "intake_jurisdiction",
]
RARE_FIELDS = ["breed", "color", "intake_subtype"]
NUMERIC_FIELDS = ["age_years", "age_missing", "intake_month_sin", "intake_month_cos"]
UNKNOWN = "UNKNOWN"
RARE = "__RARE_OR_NEW__"


def normalize_category(values):
    """Standardise spelling format without guessing what a category means."""
    clean = values.astype("string").str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
    clean = clean.fillna(UNKNOWN)
    return clean.replace({"": UNKNOWN, "UNK": UNKNOWN, "UNSPECIFIED": UNKNOWN,
                          "NOT RECORDED": UNKNOWN})


def arrival_features(records):
    """Deterministic features only: this function learns no dataset statistics."""
    missing_columns = sorted(set(RAW_INPUT_FIELDS) - set(records.columns))
    if missing_columns:
        raise ValueError("Missing input columns: " + ", ".join(missing_columns))
    selected = records[RAW_INPUT_FIELDS].copy()
    result = pd.DataFrame(index=records.index)
    for column in CATEGORICAL_FIELDS:
        if column != "biological_sex":
            result[column] = normalize_category(selected[column])
    sex_mapping = {
        "MALE": "MALE", "NEUTERED": "MALE", "NEUTERED MALE": "MALE",
        "FEMALE": "FEMALE", "SPAYED": "FEMALE", "SPAYED FEMALE": "FEMALE",
    }
    result["biological_sex"] = normalize_category(selected["sex"]).map(sex_mapping).fillna(UNKNOWN)

    intake = pd.to_datetime(selected["intake_date"], errors="coerce", format="mixed").dt.normalize()
    if intake.isna().any():
        raise ValueError("Each prediction needs a valid intake date; do not guess this date.")
    birth = pd.to_datetime(selected["date_of_birth"], errors="coerce", format="mixed").dt.normalize()
    age = (intake - birth).dt.days / 365.25
    result["age_years"] = age.where(age >= 0)
    result["age_missing"] = result["age_years"].isna().astype(int)
    # The circle puts December next to January, instead of at opposite ends.
    angle = 2 * np.pi * (intake.dt.month - 1) / 12
    result["intake_month_sin"] = np.sin(angle)
    result["intake_month_cos"] = np.cos(angle)
    return result[CATEGORICAL_FIELDS + NUMERIC_FIELDS]


class ShelterFeatures(BaseEstimator, TransformerMixin):
    """Learn age replacements and category grouping from the fit data only."""
    def __init__(self, rare_min_count=10):
        self.rare_min_count = rare_min_count

    def fit(self, X, y=None):
        if not isinstance(self.rare_min_count, int) or self.rare_min_count < 1:
            raise ValueError("rare_min_count must be a positive integer.")
        features = arrival_features(X)
        self.global_age_median_ = float(features["age_years"].median())
        if not np.isfinite(self.global_age_median_):
            raise ValueError("Training data contains no valid ages; an age replacement cannot be learned.")
        self.age_medians_by_type_ = features.groupby("type")["age_years"].median().dropna().to_dict()
        self.kept_categories_ = {}
        for column in CATEGORICAL_FIELDS:
            frequency = features[column].value_counts()
            keep = frequency[frequency >= self.rare_min_count].index if column in RARE_FIELDS else frequency.index
            self.kept_categories_[column] = set(keep) | {UNKNOWN}
        self.fit_rows_ = len(X)
        return self

    def transform(self, X):
        check_is_fitted(self, "kept_categories_")
        features = arrival_features(X)
        replacement = features["type"].map(self.age_medians_by_type_).fillna(self.global_age_median_)
        features["age_years"] = features["age_years"].fillna(replacement)
        for column in CATEGORICAL_FIELDS:
            features[column] = features[column].where(
                features[column].isin(self.kept_categories_[column]), RARE
            )
        return features[CATEGORICAL_FIELDS + NUMERIC_FIELDS]

    def get_feature_names_out(self, input_features=None):
        return np.array(CATEGORICAL_FIELDS + NUMERIC_FIELDS, dtype=object)


class CategoryEncoder(BaseEstimator, TransformerMixin):
    """One-hot encoding with explicit slots for unavailable and new categories."""
    def fit(self, X, y=None):
        self.feature_names_in_ = np.array(X.columns, dtype=object)
        # Reserved slots do not add artificial rows. Their values are fixed constants.
        categories = [sorted(set(X[column]) | {UNKNOWN, RARE}) for column in X.columns]
        self.encoder_ = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
        self.encoder_.fit(X)
        return self

    def transform(self, X):
        check_is_fitted(self, "encoder_")
        return self.encoder_.transform(X)

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "encoder_")
        return self.encoder_.get_feature_names_out(self.feature_names_in_ if input_features is None else input_features)


def make_preprocessor(rare_min_count=10):
    """Return a NEW, unfitted pipeline, including feature preparation."""
    columns = ColumnTransformer([
        ("age", StandardScaler(), ["age_years"]),
        ("flags_and_month", "passthrough", ["age_missing", "intake_month_sin", "intake_month_cos"]),
        ("categories", CategoryEncoder(), CATEGORICAL_FIELDS),
    ], remainder="drop", sparse_threshold=0)
    return Pipeline([("features", ShelterFeatures(rare_min_count=rare_min_count)), ("columns", columns)])

'''
MODULE_PATH = RESULTS_DIR / "shelter_preprocessing.py"
MODULE_PATH.write_text(MODULE_CODE, encoding="utf-8")
if str(RESULTS_DIR) not in sys.path:
    sys.path.insert(0, str(RESULTS_DIR))
importlib.invalidate_caches()
if "shelter_preprocessing" in sys.modules:
    del sys.modules["shelter_preprocessing"]
import shelter_preprocessing as prep
print("Saved reusable preprocessing code.")
print("Selected raw input fields:", prep.RAW_INPUT_FIELDS)
print("Readable features before encoding:", prep.CATEGORICAL_FIELDS + prep.NUMERIC_FIELDS)

Saved reusable preprocessing code.
Selected raw input fields: ['type', 'breed', 'color', 'sex', 'date_of_birth', 'intake_date', 'intake_type', 'intake_subtype', 'intake_condition', 'intake_jurisdiction']
Readable features before encoding: ['type', 'breed', 'color', 'biological_sex', 'intake_type', 'intake_subtype', 'intake_condition', 'intake_jurisdiction', 'age_years', 'age_missing', 'intake_month_sin', 'intake_month_cos']


## 4. Keep inputs, answers and audit details separate

**X** contains the inputs. **y** contains the 0/1 target. Separate audit details keep the row number, animal ID and dates needed to check future validation folds.

An explicit input list prevents accidental use of outcome information. Do not replace it with "every column except the target".

In [3]:
raw_inputs = {name: data[prep.RAW_INPUT_FIELDS].copy() for name, data in records.items()}
targets = {name: data["long_stay_30"].to_numpy(dtype=np.int64) for name, data in records.items()}
for name in records:
    assert set(np.unique(targets[name])) == {0, 1}
    assert "days_in_shelter" not in raw_inputs[name]
    assert "long_stay_30" not in raw_inputs[name]
    assert "id" not in raw_inputs[name]
    print(name, "rows:", len(raw_inputs[name]), "raw input columns:", len(raw_inputs[name].columns))

train rows: 6096 raw input columns: 10
validation rows: 1163 raw input columns: 10
test rows: 1672 raw input columns: 10


## 5. Learn preprocessing from training data only

Notice that **fit_transform** appears only for training data. It learns the medians, category groups, encoding and age scale.

We show the replacement ages so your group can explain the real values instead of inventing an example. The age replacement is a middle value from training records, not a confirmed age.

In [4]:
preprocessor = prep.make_preprocessor(rare_min_count=10)
prepared = {"train": preprocessor.fit_transform(raw_inputs["train"])}
learned = preprocessor.named_steps["features"]
feature_names = preprocessor.get_feature_names_out().astype(str)
trained_state_hash = joblib.hash(preprocessor)

print("Age replacements learned from training records:")
for animal_type, age in sorted(learned.age_medians_by_type_.items()):
    print(animal_type, "->", round(age, 3), "years")
print("Overall fallback:", round(learned.global_age_median_, 3), "years")
print("Encoded training shape:", prepared["train"].shape)
train_readable = learned.transform(raw_inputs["train"])
print("Training records with estimated age:", int(train_readable["age_missing"].sum()))
print("First six readable training rows:")
print(train_readable.head(6).to_string(index=False))

Age replacements learned from training records:
CAT -> 0.252 years
DOG -> 2.001 years
OTHER -> 1.899 years
Overall fallback: 1.495 years
Encoded training shape: (6096, 193)
Training records with estimated age: 1394
First six readable training rows:
type          breed           color biological_sex intake_type   intake_subtype intake_condition intake_jurisdiction  age_years  age_missing  intake_month_sin  intake_month_cos
 DOG GREAT PYRENEES           WHITE           MALE       STRAY            FIELD          UNKNOWN              COUNTY   1.960301            0               0.0               1.0
 DOG  BORDER COLLIE     BLACK/WHITE           MALE  QUARANTINE            FIELD          HEALTHY              COUNTY   3.252567            0               0.0               1.0
 DOG       PIT BULL __RARE_OR_NEW__         FEMALE  CONFISCATE       FLD_ARREST          UNKNOWN          SANTA ROSA   0.993840            0               0.0               1.0
 DOG       PIT BULL     BLACK/WHITE        

## 6. Apply the same rules to validation and test inputs

Here we use **transform**, not fit. A validation animal's age does not change our learned median. A new category does not change the training vocabulary.

We only check that transformation succeeds and preserves the rows. We do not compare test outcomes, calculate model scores or choose features using test results.

In [5]:
prepared["validation"] = preprocessor.transform(raw_inputs["validation"])
prepared["test"] = preprocessor.transform(raw_inputs["test"])
assert joblib.hash(preprocessor) == trained_state_hash, "Transform changed fitted rules."
assert learned.fit_rows_ == len(raw_inputs["train"])
for name, matrix in prepared.items():
    assert matrix.shape == (len(raw_inputs[name]), len(feature_names))
    assert np.issubdtype(matrix.dtype, np.number)
    assert np.isfinite(matrix).all(), f"{name}: missing or infinite numeric values remain"
    print(name, "prepared shape:", matrix.shape)
assert len(set(feature_names)) == len(feature_names)
assert sum(len(data) for data in raw_inputs.values()) == 8931
print("No rows removed. Same feature columns in every split. Learned rules unchanged.")

train prepared shape: (6096, 193)
validation prepared shape: (1163, 193)
test prepared shape: (1672, 193)
No rows removed. Same feature columns in every split. Learned rules unchanged.


## 7. Check important edge cases

These are small made-up checks kept separate from the real dataset:

- missing age for a known animal type;
- a new animal type with no training median;
- a new breed label;
- a birth date after arrival;
- an unreadable intake date, which should produce a clear error.

We also change outcome fields in a copy and confirm that they cannot affect the transformed inputs. None of these test examples are added to your training records.

In [6]:
examples = raw_inputs["train"].head(3).copy()
examples["type"] = ["DOG", "NEW_ANIMAL_TYPE", "CAT"]
examples["date_of_birth"] = [None, None, "2099-01-01"]
examples["breed"] = ["NEW_BREED_LABEL", None, "  unknown  "]
examples["sex"] = ["NEUTERED", "SPAYED", "UNKNOWN"]
readable_examples = learned.transform(examples)
assert readable_examples["age_missing"].eq(1).all()
assert np.isclose(readable_examples.iloc[0]["age_years"], learned.age_medians_by_type_["DOG"])
assert np.isclose(readable_examples.iloc[1]["age_years"], learned.global_age_median_)
assert readable_examples.iloc[0]["biological_sex"] == "MALE"
assert readable_examples.iloc[1]["biological_sex"] == "FEMALE"
assert readable_examples.iloc[0]["breed"] == prep.RARE
assert readable_examples.iloc[2]["breed"] == prep.UNKNOWN
assert np.isfinite(preprocessor.transform(examples)).all()

bad_input = examples.head(1).copy()
bad_input["intake_date"] = "not-a-date"
try:
    preprocessor.transform(bad_input)
except ValueError as error:
    assert "valid intake date" in str(error)
else:
    raise AssertionError("An invalid intake date should be rejected.")

extra_columns = records["train"].head(3).copy()
extra_columns["outcome_type"] = "MADE_UP_OUTCOME"
extra_columns["days_in_shelter"] = 99999
extra_columns["long_stay_30"] = 999
np.testing.assert_allclose(preprocessor.transform(extra_columns), prepared["train"][:3])
assert joblib.hash(preprocessor) == trained_state_hash
fresh_pipeline = clone(preprocessor)
assert not hasattr(fresh_pipeline.named_steps["features"], "age_medians_by_type_")
print("Missing-age, new-category, invalid-date, leakage and fresh-pipeline checks passed.")

Missing-age, new-category, invalid-date, leakage and fresh-pipeline checks passed.


## 8. Save a record of our column decisions

This file explains why each original source field is used or left out. It supports the viva and report. Keep the notes accurate if your group changes a decision.

The 10 source fields become **12 readable features** before one-hot encoding: eight categorical fields, age, an age-missing flag and two month values. Encoding then creates additional numeric columns. The printed shape shows the actual number.

In [7]:
decisions = {
    "type": ("Use", "Animal type; also selects a training-only replacement age."),
    "breed": ("Use and group rare labels", "Breed as recorded; retain uncertainty about historical updates."),
    "color": ("Use and group rare labels", "Coat colour as recorded; no causal claim."),
    "sex": ("Derive biological sex", "Discard neuter/spay status, which could change after arrival."),
    "size": ("Exclude for first pipeline", "Mixed physical-size/life-stage labels; arrival-time value not verified."),
    "date_of_birth": ("Derive age and missing flag", "Approximate age at arrival; invalid or absent values are unavailable."),
    "intake_date": ("Derive month sine/cosine", "Calendar timing known at arrival; original date used for audit/splitting."),
    "intake_type": ("Use", "Recorded reason for arrival."),
    "intake_subtype": ("Use and group rare labels", "More detailed arrival reason."),
    "intake_condition": ("Use with UNKNOWN category", "Source explicitly describes condition at intake; often unknown."),
    "intake_jurisdiction": ("Use", "Area responsible for intake; different from destination."),
    "name": ("Exclude", "Name/identifier, sometimes assigned by shelter staff."),
    "id": ("Audit only", "Animal identity used for group-overlap checks."),
    "impound_number": ("Audit only", "Admission identifier, not an animal characteristic."),
    "kennel_number": ("Exclude", "Source describes current location, not guaranteed location at arrival."),
    "outcome_date": ("Target/audit only", "Known after arrival; would reveal later information."),
    "days_in_shelter": ("Exclude from inputs", "Final duration gives away the target."),
    "outcome_type": ("Target/audit only", "Reason for release is later information."),
    "outcome_subtype": ("Exclude", "Detailed later outcome information."),
    "outcome_condition": ("Exclude", "Condition at release, not arrival."),
    "outcome_jurisdiction": ("Exclude", "Destination information."),
    "zip_code": ("Exclude", "Source defines destination ZIP code."),
    "location": ("Exclude", "Source defines outcome-jurisdiction coordinates."),
    "intake_total": ("Exclude", "Source aggregation field, not an arrival characteristic."),
}
decision_rows = [{"source_field": field, "action": choice, "reason": reason}
                 for field, (choice, reason) in decisions.items()]
pd.DataFrame(decision_rows).to_csv(RESULTS_DIR / "column_decisions.csv", index=False)
print("Saved decisions for all 24 original source columns.")

Saved decisions for all 24 original source columns.


## 9. Save the prepared data and reusable pipeline

Main outputs:

- **prepared_arrays.npz:** numeric X arrays, y labels, feature names and row numbers, kept separately for each split.
- **preprocessing_pipeline.joblib:** the fitted preprocessing rules. It is not a trained prediction model.
- **shelter_preprocessing.py:** the helper definitions needed when loading that pipeline on another computer.
- **train_inputs.csv**, **validation_inputs.csv**, **test_inputs.csv:** selected raw inputs for building fresh pipelines during model tuning.
- **train_engineered_features.csv:** readable training features before scaling and one-hot encoding.
- **row_audit.csv:** split, identifiers, label dates and target for future group/time checks; never model inputs.
- **feature_names.csv**, **column_decisions.csv**, **preprocessing_notes.json** and **requirements_preprocessing.txt:** evidence and reproduction details.
- **README.txt:** instructions for using these files later.

Keep the whole step5_results folder when sharing. Rerunning this cell replaces generated files there; it does not change Steps 1-4.

**For future tuning:** put a fresh preprocessing pipeline inside each model pipeline and fit it on the current training fold. Do not run cross-validation on globally preprocessed arrays. The saved arrays are for our fixed train/validation/test split, not automatic row-random cross-validation.

In [8]:
arrays = {"feature_names": feature_names}
for name in ["train", "validation", "test"]:
    arrays[f"X_{name}"] = prepared[name]
    arrays[f"y_{name}"] = targets[name]
    arrays[f"row_numbers_{name}"] = records[name]["source_row_number"].to_numpy(dtype=np.int64)
    raw_inputs[name].to_csv(RESULTS_DIR / f"{name}_inputs.csv", index=False)
np.savez_compressed(RESULTS_DIR / "prepared_arrays.npz", **arrays)
train_readable.to_csv(RESULTS_DIR / "train_engineered_features.csv", index=False)
pd.DataFrame({"feature_name": feature_names}).to_csv(RESULTS_DIR / "feature_names.csv", index=False)

audit_columns = ["source_row_number", "id", "impound_number", "intake_date", "label_known_date", "long_stay_30"]
audit = pd.concat([data[audit_columns].assign(split=name) for name, data in records.items()], ignore_index=True)
audit.to_csv(RESULTS_DIR / "row_audit.csv", index=False)
joblib.dump(preprocessor, RESULTS_DIR / "preprocessing_pipeline.joblib")

versions = {"pandas": pd.__version__, "numpy": np.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "joblib": joblib.__version__}
(RESULTS_DIR / "requirements_preprocessing.txt").write_text(
    "\n".join(f"{package}=={version}" for package, version in versions.items()) + "\n", encoding="utf-8")
preprocessing_notes = {
    "source_url": split_notes["source_url"], "source_sha256_by_split": source_hashes,
    "raw_input_fields": prep.RAW_INPUT_FIELDS,
    "readable_features": prep.CATEGORICAL_FIELDS + prep.NUMERIC_FIELDS,
    "encoded_feature_count": len(feature_names),
    "rows": {name: len(data) for name, data in records.items()},
    "rows_removed_in_step5": 0,
    "training_rows_with_estimated_age": int(train_readable["age_missing"].sum()),
    "age_medians_by_type": {k: float(v) for k, v in learned.age_medians_by_type_.items()},
    "overall_age_median": float(learned.global_age_median_),
    "rare_min_count": learned.rare_min_count,
    "rare_grouping_fields": prep.RARE_FIELDS,
    "fit_data": "Training inputs only; no target used to learn preprocessing.",
    "fitted_state_hash": trained_state_hash,
    "libraries": versions, "python_version": sys.version.split()[0],
    "status": "Preprocessing implemented and checked; no prediction model trained or evaluated.",
    "limitations": [
        "Arrival-time availability of mutable attributes cannot be proven from this snapshot.",
        "Size omitted; biological sex excludes neuter/spay status but record timing remains uncertain.",
        "Birth dates are approximate; imputed ages are estimates, including mixed-species OTHER.",
        "Rare grouping threshold 10 is an initial choice; revisit only through training/validation.",
        "Ongoing-label and retrospective evaluation assumptions from earlier notebooks still apply.",
        "A reserved new-category slot has no directly learned evidence if absent from training."
    ],
    "documentation": ["https://scikit-learn.org/stable/common_pitfalls.html",
                      "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html"]
}
(RESULTS_DIR / "preprocessing_notes.json").write_text(json.dumps(preprocessing_notes, indent=2), encoding="utf-8")
readme = """KND_04 - Step 5 results

Keep this folder together. Tested versions are in requirements_preprocessing.txt.
This is preprocessing only: no trained classifier or measured model accuracy exists yet.

The fitted pipeline requires shelter_preprocessing.py to be importable.
To reload in a notebook:
    from pathlib import Path
    import sys, joblib
    folder = Path('step5_results').resolve()
    sys.path.insert(0, str(folder))
    pipeline = joblib.load(folder / 'preprocessing_pipeline.joblib')
    numeric_inputs = pipeline.transform(new_arrival_dataframe)

new_arrival_dataframe must contain the 10 raw input columns in preprocessing_notes.json.
Birth date may be missing, but intake date must be valid.
Use the same dependency versions when loading the saved pipeline.

For fresh training or time/group-aware tuning, import make_preprocessor from
shelter_preprocessing and put it before the classifier in a new sklearn Pipeline.
Fit inside each training fold. Do not fit on validation/test inputs, and do not
cross-validate already transformed whole-training arrays.

prepared_arrays.npz can be opened with numpy.load(..., allow_pickle=False).
X_train, X_validation and X_test share the feature_names order.
y_train, y_validation and y_test are separate targets, aligned by row_numbers_*.
row_audit.csv contains bookkeeping/target columns that must not become model inputs.

Changing the preprocessing rules requires rerunning this notebook and subsequent
model steps. Keep split dates fixed and reserve test evaluation until choices are final.
"""
(RESULTS_DIR / "README.txt").write_text(readme, encoding="utf-8")
print("Saved prepared data, reusable code, fitted pipeline and decision notes.")

Saved prepared data, reusable code, fitted pipeline and decision notes.


## 10. Reopen the saved files and verify

Saving without checking is not enough. We reload the arrays and pipeline and compare them with the results in memory.

The expected row counts remain **6,096 training**, **1,163 validation** and **1,672 test**. There are **8,931 records in total after our current screening and preparation**, above the assignment's 5,000-record requirement mentioned in your feedback.

In [9]:
with np.load(RESULTS_DIR / "prepared_arrays.npz", allow_pickle=False) as saved:
    for name in records:
        np.testing.assert_allclose(saved[f"X_{name}"], prepared[name])
        np.testing.assert_array_equal(saved[f"y_{name}"], targets[name])
        np.testing.assert_array_equal(saved[f"row_numbers_{name}"], records[name]["source_row_number"])
    np.testing.assert_array_equal(saved["feature_names"], feature_names)

restored = joblib.load(RESULTS_DIR / "preprocessing_pipeline.joblib")
np.testing.assert_allclose(restored.transform(raw_inputs["train"].head(5)), prepared["train"][:5])
for name in records:
    assert hashlib.sha256((STEP3_DIR / f"{name}_records.csv").read_bytes()).hexdigest() == source_hashes[name]
print("All preprocessing and saved-file checks passed.")
print("Training:", prepared["train"].shape)
print("Validation:", prepared["validation"].shape)
print("Test:", prepared["test"].shape)
print("Original split files unchanged. No prediction model trained yet.")

All preprocessing and saved-file checks passed.
Training: (6096, 193)
Validation: (1163, 193)
Test: (1672, 193)
Original split files unchanged. No prediction model trained yet.


## Group work and viva preparation

- **Member 1:** explain the allowed inputs, derived target separation and omitted fields.
- **Member 2:** connect the Notebook 4 findings to each preprocessing decision.
- **Member 3:** explain age replacement, unknown/rare categories, one-hot encoding and scaling.
- **Member 4:** explain fit versus transform, saved files and leakage checks.

Everyone should rerun the notebook and explain the complete process. Record actual contributions rather than claiming all supplied code was written independently.

### Simple viva explanation

"We calculated age at arrival and treated missing or impossible ages as unavailable. We filled them using training-data medians and added a flag to show which ages were estimated. We grouped rare categories, encoded the inputs and scaled age. We learned these rules only from training data and reused them unchanged for validation and testing. We kept the records and excluded fields that could reveal the outcome."

### Can we explain these five decisions?

1. Why does an unknown dog age use a training median rather than an average from the full dataset?
2. Why do we keep an age-missing flag after filling the age?
3. Why are NEUTERED and MALE mapped to the same biological-sex category?
4. Why is a label such as DOG encoded rather than assigned an arbitrary rank?
5. Why must a new preprocessing pipeline be fitted inside each training fold during tuning?

### Progress Evaluation 1

The notebooks now provide an implemented workflow through preprocessing. Your group still needs to review the decisions, record contributions, keep approval evidence and practise explaining the code and findings. The scope follows the assignment; marks depend on your own understanding and evaluation.

### Next step

**Notebook 6: compare at least four models.** We will use validation data for comparison, preserve the fixed temporal/group split, and keep final test scores reserved until model choices are fixed.

Technical references: [scikit-learn guidance on leakage and consistent preprocessing](https://scikit-learn.org/stable/common_pitfalls.html) and [OneHotEncoder documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).